# 🎲 Análisis Exploratorio de Datos (EDA) — FIDE Chess Dataset

**Evaluación 1 — AD 1.1: Ingestión y Exploración**

Este notebook cubre:
1. Carga de los 4 datasets desde el catálogo Kedro
2. Perfil inicial: forma, tipos, head(), describe(), info()
3. Análisis de valores nulos y duplicados
4. Distribuciones de variables clave (ELO, edad, género, títulos)
5. Correlaciones y heatmaps
6. Conclusiones sobre calidad de datos

In [ ]:
%load_ext kedro.ipython

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualizaciones
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 1. Carga de Datos desde el Catálogo Kedro

In [ ]:
# Cargar datasets crudos
players = catalog.load('fide_players')
ratings_2019 = catalog.load('fide_ratings_2019')
ratings_2020 = catalog.load('fide_ratings_2020')
ratings_2021 = catalog.load('fide_ratings_2021')

print(f'Players:      {players.shape}')
print(f'Ratings 2019: {ratings_2019.shape}')
print(f'Ratings 2020: {ratings_2020.shape}')
print(f'Ratings 2021: {ratings_2021.shape}')

## 2. Perfil Inicial de los Datos

In [ ]:
print('=== PLAYERS ===')
print(f'Dimensiones: {players.shape}')
print(f'\nColumnas y tipos:')
print(players.dtypes)
print(f'\nPrimeras 5 filas:')
display(players.head())

In [ ]:
print('=== Estadísticas Descriptivas: Players ===')
display(players.describe(include='all'))

In [ ]:
players.info()

In [ ]:
print('=== RATINGS 2021 (ejemplo) ===')
print(f'Dimensiones: {ratings_2021.shape}')
display(ratings_2021.head())
display(ratings_2021.describe())

## 3. Análisis de Calidad: Nulos y Duplicados

In [ ]:
# Gráfico de nulos por dataset
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

datasets = {
    'Players': players,
    'Ratings 2019': ratings_2019,
    'Ratings 2020': ratings_2020,
    'Ratings 2021': ratings_2021,
}

for ax, (name, df) in zip(axes, datasets.items()):
    null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=True)
    null_pct.plot.barh(ax=ax, color='coral')
    ax.set_title(f'% Nulos \u2014 {name}')
    ax.set_xlabel('% Nulos')

plt.tight_layout()
plt.show()

# Resumen numérico
for name, df in datasets.items():
    n_dup = df.duplicated().sum()
    n_null = df.isnull().sum().sum()
    print(f'{name:15s} | Filas: {len(df):>10,} | Nulos totales: {n_null:>10,} | Duplicados: {n_dup:>6,}')

## 4. Distribuciones de Variables Clave

In [ ]:
# Distribución de género
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gender_counts = players['gender'].value_counts()
axes[0].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2'))
axes[0].set_title('Distribución por Género')

# Distribución de títulos
title_counts = players['title'].value_counts().head(10)
title_counts.plot.bar(ax=axes[1], color=sns.color_palette('viridis', len(title_counts)))
axes[1].set_title('Top 10 Títulos FIDE')
axes[1].set_ylabel('Cantidad de Jugadores')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de año de nacimiento
fig, ax = plt.subplots(figsize=(14, 5))
players['yob'].dropna().astype(int).hist(bins=80, ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Distribución del Año de Nacimiento de Jugadores FIDE')
ax.set_xlabel('Año de Nacimiento')
ax.set_ylabel('Cantidad')
plt.show()

In [ ]:
# Distribución de ELO Estándar (2021)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (year, df) in zip(axes, [
    ('2019', ratings_2019), ('2020', ratings_2020), ('2021', ratings_2021)
]):
    elo = df['rating_standard'].dropna()
    ax.hist(elo, bins=80, color='darkorange', edgecolor='white', alpha=0.8)
    ax.axvline(elo.mean(), color='red', linestyle='--', label=f'Media: {elo.mean():.0f}')
    ax.axvline(2000, color='green', linestyle='--', label='Umbral Experto (2000)')
    ax.set_title(f'Distribución ELO Estándar {year}')
    ax.set_xlabel('Rating Estándar')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots para detectar outliers en ratings 2021
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, ['rating_standard', 'rating_rapid', 'rating_blitz']):
    data = ratings_2021[col].dropna()
    ax.boxplot(data, vert=True)
    ax.set_title(f'Boxplot: {col}')
    ax.set_ylabel('Rating')

plt.tight_layout()
plt.show()

In [ ]:
# Top 15 federaciones por cantidad de jugadores
fig, ax = plt.subplots(figsize=(14, 6))
top_fed = players['federation'].value_counts().head(15)
top_fed.plot.bar(ax=ax, color=sns.color_palette('coolwarm', len(top_fed)))
ax.set_title('Top 15 Federaciones por Cantidad de Jugadores')
ax.set_ylabel('Jugadores')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 5. Correlaciones

In [ ]:
# Correlación entre los 3 tipos de rating (2021)
numeric_ratings = ratings_2021[['rating_standard', 'rating_rapid', 'rating_blitz']].dropna()

fig, ax = plt.subplots(figsize=(8, 6))
corr = numeric_ratings.corr()
sns.heatmap(corr, annot=True, cmap='RdYlBu_r', vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax, fmt='.3f')
ax.set_title('Correlación entre Tipos de Rating (2021)')
plt.tight_layout()
plt.show()

## 6. Conclusiones del EDA

**Hallazgos principales:**

1. **Volumen de datos:** El dataset contiene ~433K jugadores y ~12M registros de ratings.
2. **Nulos:** La columna `title` tiene un alto porcentaje de nulos (la mayoría de jugadores no tiene título FIDE). Las columnas `rating_rapid` y `rating_blitz` también presentan nulos significativos.
3. **Distribuciones:** El rating estándar sigue una distribución aproximadamente normal con media alrededor de 1500-1600.
4. **Género:** Existe un desbalance significativo, con predominancia masculina.
5. **Correlaciones:** Los tres tipos de rating (standard, rapid, blitz) están altamente correlacionados.
6. **Outliers:** Se detectan valores atípicos en los extremos de los ratings (muy bajos o muy altos).

Estos hallazgos justifican las transformaciones que se aplican en los pipelines de limpieza y transformación.